In [ ]:
# --- Manual backpropagation (practice decomposing loss.backward()) ---

In [ ]:
# --- Boilerplate START ---

In [7]:
import torch
import torch.nn.functional as F
import random
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [8]:
with open("names.txt", "r") as f:
    words = f.read().splitlines()

chars = sorted((set("".join(words))))
stoi = {c:i + 1 for i, c in enumerate(chars)} # {".":0, "a":1, ...}
stoi["."] = 0
itos = {i:c for c, i in stoi.items()} # {0:".", 1:"a", ...}

In [9]:
block_size = 3

def build_dataset(words):
    X, Y = [], []
    for word in words:
        context = [0] * block_size
        for ch in word + ".":
            idx = stoi[ch]
            X.append(context)
            Y.append(idx)
            context = context[1:] + [idx]
    return torch.tensor(X), torch.tensor(Y)

random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [11]:
# Utility function for comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [13]:
n_embd = 10
n_hidden = 200
vocab_size = len(itos)

# Embedding model (lookup table)
C = torch.randn(vocab_size, n_embd)

# Layer 1
W1 = torch.randn(n_embd * block_size, n_hidden) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden) * 0.1 # b1 is just for fun (useless because of BN)

# Layer 2
W2 = torch.randn(n_hidden, vocab_size) * 0.1 + 1.0
b2 = torch.randn(vocab_size) * 0.1

# BatchNorm
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
  p.requires_grad = True

sum(p.nelement() for p in parameters) # Total no. of parameters

# NOTE: weight initializations are set unconventionally to more easily detect incorrect backpropagation implements!

12297

In [ ]:
batch_size = 32
n = batch_size # Shorter variable for convenience
ix = torch.randint(0, Xtr.shape[0], (batch_size,)) # Choose 32 random row indices out of ~182k
Xb, Yb = Xtr[ix], Ytr[ix] # Xb: (32, 3), Yb: (32,)

In [15]:
# Forward pass ("chunkated" into smaller steps that are possible to backprop one at a time)

emb = C[Xb] # Embed the characters into vectors (32, 3, 10)
embcat = emb.view(emb.shape[0], -1) # Concatenate the vectors (32, 30)

# Linear layer 1
hprebn = embcat @ W1 + b1 # Hidden layer pre-activation

# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias

# Non-linearity
h = torch.tanh(hpreact) # Hidden layer

# Linear layer 2
logits = h @ W2 + b2 # Output layer

# Cross-entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # Subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # (1.0 / counts_sum) makes backprop not bit exact
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # AFAIK there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.5770, grad_fn=<NegBackward0>)

In [ ]:
# --- Boilerplate END ---

In [129]:
# Exercise 1: backprop through the whole thing manually, 
# backpropagating through exactly all of the variables 
# as they are defined in the forward pass above, one by one

# -----------------
# YOUR CODE HERE :)
# -----------------

dlogprobs = torch.zeros_like(logprobs) # zeros_like is just zeros but automatically copies logprobs' dimensions
dlogprobs[range(n), Yb] = -1.0/n # Only want -1/n in the selected indices, otherwise 0
dprobs = dlogprobs * (1.0 / probs) # dlog(x)dx = 1/x
dcounts_sum_inv = (dprobs * counts).sum(1, keepdim=True) # Sum since dprobs * counts is (32, 27) but counts_sum_inv is (32, 1)
dcounts_sum = dcounts_sum_inv * -(1.0 / counts_sum ** 2)
dcounts = dprobs * counts_sum_inv # First part (from probs)
dcounts += dcounts_sum * torch.ones_like(counts) # Second part (from counts_sum)
dnorm_logits = dcounts * norm_logits.exp()
dlogit_maxes = (dnorm_logits * -1).sum(1, keepdim=True) # Sum to make (32, 27) -> (32, 1)
dlogits = dnorm_logits.clone()
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes # Add dlogit_maxes back to the max indices (F.one_hot creates one-hot vectors with length 27 and 1s at max indices)
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0, keepdim=True)
dhpreact = dh * (1 - torch.tanh(hpreact) ** 2)
dbngain = (dhpreact * bnraw).sum(0, keepdim=True) # Sum to get (1, 200)
dbnbias = dhpreact.sum(0, keepdim=True) # Sum to get (1, 200)
dbnraw = dhpreact * bngain
dbnvar_inv = (dbnraw * bndiff).sum(0, keepdim=True) # Sum to get (1, 200)
dbnvar = dbnvar_inv * (-0.5 * (bnvar + 1e-5)**-1.5) # bnvar is the squared var!
dbndiff2 = dbnvar *  (1.0/(n - 1)) * torch.ones_like(bndiff2)
dbndiff = dbnraw * bnvar_inv # First part (from bnraw)
dbndiff += dbndiff2 * 2.0 * bndiff # Second part (from bndiff2)
dbnmeani = -dbndiff.sum(0, keepdim=True) # Sum to get (1, 200)
dhprebn = dbndiff.clone() # First part (from bndiff)
dhprebn += dbnmeani * (torch.ones_like(hprebn) * 1.0 / n) # Second part (from bnmeani)
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn 
db1 = dhprebn.sum(0)
demb = dembcat.view(emb.shape) # No number change, just dimensions(32, 30) -> (32, 3, 10)
dC = torch.zeros_like(C) # dC = how loss changes when rows of C change, so for dC_i, add up all the gradients related to row i in emb
for i in range(Xb.shape[0]):
    for t in range(Xb.shape[1]):
        dC[Xb[i, t]] += demb[i, t] # Same row could have been used many times

cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff:

In [130]:
# Exercise 2: backprop through cross_entropy but all in one go
# to complete this challenge look at the mathematical expression of the loss,
# take the derivative, simplify the expression, and just write it out

# forward pass

# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

3.5770397186279297 diff: 4.76837158203125e-07


In [ ]:
# backward pass

# -----------------
# YOUR CODE HERE :)
dlogits = F.softmax(logits, 1)
dlogits[range(n), Yb] -= 1
dlogits /= n # Average loss over n
# -----------------

cmp('logits', dlogits, logits) # I can only get approximate to be true, my maxdiff is 6e-9

torch.Size([32, 27])
logits          | exact: False | approximate: True  | maxdiff: 5.122274160385132e-09


In [136]:
# Exercise 3: backprop through batchnorm but all in one go
# to complete this challenge look at the mathematical expression of the output of batchnorm,
# take the derivative w.r.t. its input, simplify the expression, and just write it out
# BatchNorm paper: https://arxiv.org/abs/1502.03167

# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# now:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

max diff: tensor(4.7684e-07, grad_fn=<MaxBackward1>)


In [138]:
# backward pass

# before we had:
# dbnraw = bngain * dhpreact
# dbndiff = bnvar_inv * dbnraw
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
# dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
# dbndiff += (2*bndiff) * dbndiff2
# dhprebn = dbndiff.clone()
# dbnmeani = (-dbndiff).sum(0)
# dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)

# calculate dhprebn given dhpreact (i.e. backprop through the batchnorm)
# (you'll also need to use some of the variables from the forward pass up above)

# -----------------
# YOUR CODE HERE :)
dhprebn = bngain * bnvar_inv / n * (
    n * dhpreact 
    - n / (n - 1) * bnraw * (dhpreact * bnraw).sum(0)
    - dhpreact.sum(0)
)
# -----------------

cmp('hprebn', dhprebn, hprebn) # I can only get approximate to be true, my maxdiff is 9e-10

hprebn          | exact: False | approximate: True  | maxdiff: 9.313225746154785e-10


In [152]:
# Exercise 4: putting it all together!
# Train the MLP neural net with your own backward pass

# init
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

# same optimization as last time
max_steps = 200000
batch_size = 32
n = batch_size # convenience
lossi = []

# use this context manager for efficiency once your backward pass is written
with torch.no_grad():

  # kick off optimization
  for i in range(max_steps):

    # minibatch construct
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

    # forward pass
    emb = C[Xb] # embed the characters into vectors
    embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
    # Linear layer
    hprebn = embcat @ W1 + b1 # hidden layer pre-activation
    # BatchNorm layer
    # -------------------------------------------------------------
    bnmean = hprebn.mean(0, keepdim=True)
    bnvar = hprebn.var(0, keepdim=True, unbiased=True)
    bnvar_inv = (bnvar + 1e-5)**-0.5
    bnraw = (hprebn - bnmean) * bnvar_inv
    hpreact = bngain * bnraw + bnbias
    # -------------------------------------------------------------
    # Non-linearity
    h = torch.tanh(hpreact) # hidden layer
    logits = h @ W2 + b2 # output layer
    loss = F.cross_entropy(logits, Yb) # loss function

    # manual backprop! #swole_doge_meme
    # -----------------
    # Cross-entropy loss
    dlogits = F.softmax(logits, 1)
    dlogits[range(n), Yb] -= 1
    dlogits /= n

    # Linear layer 2
    dh = dlogits @ W2.T
    dW2 = h.T @ dlogits
    db2 = dlogits.sum(0)

    # Nonlinearity  
    dhpreact = dh * (1 - torch.tanh(hpreact) ** 2)

    # BatchNorm
    dhprebn = bngain * bnvar_inv / n * (
        n * dhpreact 
        - n / (n - 1) * bnraw * (dhpreact * bnraw).sum(0)
        - dhpreact.sum(0)
    )
    dbngain = (dhpreact * bnraw).sum(0, keepdim=True)
    dbnbias = dhpreact.sum(0, keepdim=True)

    # Linear layer 1
    dembcat = dhprebn @ W1.T
    dW1 = embcat.T @ dhprebn 
    db1 = dhprebn.sum(0)

    # Embedding
    demb = dembcat.view(emb.shape)
    dC = torch.zeros_like(C)
    for j in range(Xb.shape[0]):
        for t in range(Xb.shape[1]):
            dC[Xb[j, t]] += demb[j, t]

    grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
    # -----------------

    # update
    lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
    for p, grad in zip(parameters, grads):
      # p.data += -lr * p.grad # old way of cheems doge (using PyTorch grad from .backward())
      p.data += -lr * grad # new way of swole doge

    # track stats
    if i % 10000 == 0: # print every once in a while
      print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())


12297
      0/ 200000: 3.7465
  10000/ 200000: 2.4596
  20000/ 200000: 2.1476
  30000/ 200000: 2.0116
  40000/ 200000: 2.5629
  50000/ 200000: 2.2629
  60000/ 200000: 2.2875
  70000/ 200000: 2.1225
  80000/ 200000: 2.5764
  90000/ 200000: 2.0641
 100000/ 200000: 2.0552
 110000/ 200000: 2.2599
 120000/ 200000: 2.4896
 130000/ 200000: 2.2732
 140000/ 200000: 1.9356
 150000/ 200000: 2.3237
 160000/ 200000: 1.9943
 170000/ 200000: 1.9470
 180000/ 200000: 1.7608
 190000/ 200000: 1.9944


In [153]:
# calibrate the batch norm at the end of training

with torch.no_grad():
  # pass the training set through
  emb = C[Xtr]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 + b1
  # measure the mean/std over the entire training set
  bnmean = hpreact.mean(0, keepdim=True)
  bnvar = hpreact.var(0, keepdim=True, unbiased=True)

In [154]:
# evaluate train and val loss

@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 + b1
  hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.070263385772705
val 2.1194634437561035


In [ ]:
# Karpathy's numbers:
# train 2.0718822479248047
# val 2.1162495613098145

In [155]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # forward pass
      emb = C[torch.tensor([context])] # (1,block_size,d)      
      embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
      hpreact = embcat @ W1 + b1
      hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
      h = torch.tanh(hpreact) # (N, n_hidden)
      logits = h @ W2 + b2 # (N, vocab_size)
      # sample
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

carlah.
amoriell.
kimri.
reigh.
skanden.
jazhitladelynn.
jareei.
ner.
kiah.
maiiv.
kaleigh.
ham.
joce.
quinn.
shon.
marianni.
watthonielrysia.
kael.
druan.
edde.


# Makemore 4 — forward pass math

Indices: $i$ = example in batch ($n=32$), $t$ = context position ($T=3$), $j$ = hidden unit (200), $k$ = vocab char (27).

## Embedding

$$
\text{emb}_{i,t} = C[X_{i,t}] \qquad (n, 3, 10)
$$

$$
x_i = \text{concat}(\text{emb}_{i,1}, \text{emb}_{i,2}, \text{emb}_{i,3}) \qquad \text{embcat}: (n, 30)
$$

## Linear 1

$$
z = xW_1 + b_1 \qquad \text{hprebn}: (n, 200)
$$

## BatchNorm (stats over batch dim $i$, per hidden unit $j$)

$$
\mu_j = \frac{1}{n}\sum_i z_{ij} \qquad \text{bnmeani}: (1, 200)
$$

$$
d_{ij} = z_{ij} - \mu_j \qquad \text{bndiff}: (n, 200)
$$

$$
d^2_{ij} = d_{ij}^2 \qquad \text{bndiff2}: (n, 200)
$$

$$
\sigma^2_j = \frac{1}{n-1}\sum_i d^2_{ij} \qquad \text{bnvar}: (1, 200)
$$

$$
r_j = (\sigma^2_j + \epsilon)^{-1/2} \qquad \text{bnvar\_inv}: (1, 200)
$$

$$
\hat z_{ij} = d_{ij} \cdot r_j \qquad \text{bnraw}: (n, 200)
$$

$$
u_{ij} = \gamma_j \hat z_{ij} + \beta_j \qquad \text{hpreact}: (n, 200)
$$

($\gamma$ = bngain, $\beta$ = bnbias)

## Nonlinearity

$$
h = \tanh(u) \qquad (n, 200)
$$

## Linear 2

$$
\ell = hW_2 + b_2 \qquad \text{logits}: (n, 27)
$$

## Softmax + cross-entropy

$$
m_i = \max_k \ell_{ik} \qquad \text{logit\_maxes}: (n, 1)
$$

$$
\ell'_{ik} = \ell_{ik} - m_i \qquad \text{norm\_logits}: (n, 27)
$$

$$
c_{ik} = e^{\ell'_{ik}} \qquad \text{counts}: (n, 27)
$$

$$
S_i = \sum_k c_{ik} \qquad \text{counts\_sum}: (n, 1)
$$

$$
S^{-1}_i = \frac{1}{S_i} \qquad \text{counts\_sum\_inv}: (n, 1)
$$

$$
p_{ik} = c_{ik} \cdot S^{-1}_i \qquad \text{probs}: (n, 27)
$$

$$
\lambda_{ik} = \log p_{ik} \qquad \text{logprobs}: (n, 27)
$$

$$
L = -\frac{1}{n}\sum_i \lambda_{i, y_i} \qquad \text{loss}: \text{scalar}
$$